DATA PROCESSING

In [1]:
# Import packages
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors

In [4]:
#loading
data = pd.read_csv('train.csv')  

In [6]:
#Get dataset size
print(f'Shape of train.csv: {data.shape}')

Shape of train.csv: (7973, 7)


In [26]:
#select subset of train.csv with a Tc value attached
train_with_Tc = train[train['Tc'].notnull()] #subset of train dataframe with Tc values

print(f"Number of SMILES with Tc values in train.csv: {len(train_with_Tc)}")

dataset1_with_Tc = dataset1[dataset1['TC_mean'].notnull()]

print(f"Number of SMILES with Tc values in dataset1.csv: {len(dataset1_with_Tc)}")



Number of SMILES with Tc values in train.csv: 737
Number of SMILES with Tc values in dataset1.csv: 874


In [33]:
#Concatenate train_with_Tc and dataset1_with_Tc for model training
combined_data = pd.concat([train_with_Tc, dataset1_with_Tc], ignore_index=True)

#set rows with TC_mean values as Tc values in combined_data
combined_data['Tc'] = combined_data.apply(lambda row: row['TC_mean'] if pd.notnull(row['TC_mean']) else row['Tc'], axis=1)

In [34]:
combined_data.head()

,id,SMILES,Tg,FFV,Tc,Density,Rg,TC_mean
0,87817.0,*CC(*)c1ccccc1C(=O)OCCCCCC,NaN,0.374645,0.205667,NaN,NaN,NaN
1,2986007.0,*c1ccc(-c2ccc3c(c2)C(CCCCCCC#N)(CCCCCCC#N)c2cc...,NaN,0.402397,0.487000,0.901123,28.682441,NaN
2,3013292.0,*CC(*)c1ccc(C(=O)O)c(C(=O)O)c1,NaN,NaN,0.171000,1.184354,13.534248,NaN
3,6645418.0,*CCCCCNC(=O)CCCCC(=O)N*,NaN,0.332741,0.327000,NaN,NaN,NaN
4,7687820.0,*CCCCCCCCCCCCCCCCCCNC(=O)NCCCCCCNC(=O)N*,NaN,NaN,0.383000,NaN,NaN,NaN


In [22]:
# A function to canonicalize isomeric SMILES
def canonicalize_smiles(smiles, isomeric:bool=True):
    """
    Convert any SMILES to its canonical form.

    Args:
        smiles (str): Input SMILES string

    Returns:
        str: Canonical SMILES
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  # Invalid SMILES
    return Chem.MolToSmiles(mol, isomericSmiles=isomeric)
